In [1]:
using Markdown
using InteractiveUtils
using LinearAlgebra

In [2]:
n = 100
x_actual :: Vector{Float64} = [(-1)^(i - 1) for i in 1:n]
A :: Matrix{Float64} = zeros(n, n)

for i in 1:n
	A[i, i] = 2

	if i != 1
		A[i, i - 1] = 1
	end

	if i != n
		A[i, i + 1] = 1
	end
end

guess = ones(n)
	
b = zeros(n)
b[1] = 1
b[n] = -1

-1

In [3]:
function applyA(x)
	if length(x) != n
		throw("Size Mismatch")
	end
	
	Ax = 2x
	for i in 1:n
		if i != 1
			Ax[i] += x[i - 1]
		end

		if i != n
			Ax[i] += x[i + 1]
		end
	end
	return Ax
end

applyA (generic function with 1 method)

In [4]:
function jacobi(A :: Matrix{Float64}, b::Vector{Float64}, x0 :: Vector{Float64}, tolerance::Float64, useForward::Bool = true, x_actual::Vector{Float64} = [0.0])
	x = x0
	iterations = 0

    n = size(A, 1)
	
	while (useForward && norm(x_actual - x, Inf) > tolerance) || (!useForward && norm(b - A * x, Inf) > tolerance)
		x1 = copy(b)
		
		for i = 1:n
			for j = 1:n
				if i != j
					x1[i] -= A[i, j] * x[j]
				end
			end

			x1[i] /= A[i, i]
		end


		x = x1
		iterations += 1
	end

	return (x, iterations)
end

jacobi (generic function with 3 methods)

In [5]:
function gauss_sidel(A :: Matrix{Float64}, b::Vector{Float64}, x0 :: Vector{Float64}, tolerance::Float64,  useForward::Bool = true, x_actual::Vector{Float64} = [0.0])
	x = x0
	iterations = 0
    n = size(A, 1)
	
	while (useForward && norm(x_actual - x, Inf) > tolerance) || (!useForward && norm(b - A * x, Inf) > tolerance)
		x1 = copy(b)
		
		for i in 1:n
			for j in 1:n
				if i < j
					x1[i] -= A[i, j] * x[j]
				elseif i > j
					x1[i] -= A[i, j] * x1[j]
				end
			end


			x1[i] /= A[i, i]
		end


		x = x1
		iterations += 1
	end

	return (x, iterations)
end

gauss_sidel (generic function with 3 methods)

In [6]:
function sor(A :: Matrix{Float64}, b::Vector{Float64}, w::Float64, x0 :: Vector{Float64}, x_actual::Vector{Float64}, tolerance::Float64)
	x = x0
	iterations = 0
	
	while norm(x_actual - x, Inf) > tolerance
		x1 = copy(b)
		
		for i = 1:n
			for j = 1:n
				if i < j
					x1[i] -= A[i, j] * x[j]
				elseif i > j
					x1[i] -= A[i, j] * x1[j]
				end
			end

			x1[i] /= A[i, i]
		end


		x = w * x1 + x * (1 - w)
		iterations += 1
	end

	return (x, iterations)
end

sor (generic function with 1 method)

In [7]:
x_jacobi, jacobi_iterations  = jacobi(A, b, guess, 0.5 * 10^(-3), true, x_actual) 

print("Approximate solution: $(x_jacobi)")

Markdown.parse("""
	## 2.5 Computer problem 2
	Number of steps: $(jacobi_iterations)
	
	Backward Error: $(norm(b - applyA(x_jacobi), Inf))

	
	""")

Approximate solution: [1.0, -0.9999689238511333, 1.0, -0.9999379679298495, 1.0, -0.9999072519985943, 1.0, -0.9998768948913388, 1.0, -0.9998470140538332, 1.0, -0.9998177250892334, 1.0, -0.9997891413108533, 1.0, -0.9997613733037785, 1.0, -0.9997345284970339, 1.0, -0.9997087107479621, 1.0, -0.9996840199404184, 1.0, -0.9996605515983407, 1.0, -0.9996383965161851, 1.0, -0.9996176404076609, 1.0, -0.9995983635741194, 1.0, -0.9995806405938849, 1.0, -0.999564540033725, 1.0, -0.9995501241835789, 1.0, -0.9995374488155706, 1.0, -0.9995265629682362, 1.0, -0.9995175087568045, 1.0, -0.9995103212102605, 1.0, -0.9995050281358253, 1.0, -0.999501650011376, 1.0, -0.9995001999062195, 1.0, -0.9995006834305311, 1.0, -0.9995030987136493, 1.0, -0.9995074364113126, 1.0, -0.9995136797418118, 1.0, -0.9995218045509138, 1.0, -0.9995317794053106, 1.0, -0.9995435657142283, 1.0, -0.9995571178787276, 1.0, -0.9995723834681166, 1.0, -0.9995893034227955, 1.0, -0.9996078122827468, 1.0, -0.9996278384407868, 1.0, -0.999649304

## 2.5 Computer problem 2

Number of steps: 17642

Backward Error: 0.0009996001875609561


In [8]:
x_gs, gs_iterations = gauss_sidel(A, b, guess, 0.5 * 10^-3,true, x_actual)
print("Approximate solution: $(x_gs)")
Markdown.parse("""## 2.5 Computer problem 6
	Number of steps: $(gs_iterations)
	
	Backward Error: $(norm(b - applyA(x_gs), Inf))
""")

Approximate solution: [0.9999840815015355, -0.9999681937958601, 0.9999523522198026, -0.9999365720507416, 0.9999208684918861, -0.9999052566576433, 0.9998897515590854, -0.9998743680895312, 0.9998591210102546, -0.9998440249363358, 0.9998290943226664, -0.999814343450124, 0.9997997864119277, -0.999785437100189, 0.9997713091926693, -0.9997574161397587, 0.999743771151686, -0.9997303871859746, 0.9997172769351542, -0.9997044528147407, 0.999691926951496, -0.9996797111719782, 0.9996678169913934, -0.9996562556027591, 0.9996450378663901, -0.9996341742997162, 0.9996236750674412, -0.9996135499720528, 0.9996038084446919, -0.9995944595363904, 0.999585511909684, -0.9995769738306102, 0.9995688531610958, -0.9995611573517452, 0.9995538934350308, -0.999547068018896, 0.9995406872807748, -0.999534756962033, 0.999529282362837, -0.9995242683374546, 0.9995197192899904, -0.9995156391705625, 0.9995120314719204, -0.9995088992265093, 0.9995062450039824, -0.9995040709091633, 0.9995023785804591, -0.9995011691887279, 0

## 2.5 Computer problem 6

```
Number of steps: 8106

Backward Error: 4.838737724943698e-7
```


In [9]:
x_sor, sor_iterations = sor(A, b, 1.5, guess, x_actual, 0.5 * 10^-3)

print("Approximate solution: $(x_sor)")
Markdown.parse("""## 2.5 Computer problem 6b
	Number of steps: $(sor_iterations)
	
	Backward Error: $(norm(b - applyA(x_sor), Inf))
""")

Approximate solution: [0.9999840886147497, -0.9999682080085306, 0.9999523735113139, -0.9999366003936563, 0.9999209038519683, -0.9999052989939073, 0.9998898008238348, -0.9998744242284172, 0.9998591839623302, -0.9998440946341242, 0.9998291706922305, -0.9998144264111477, 0.999799875877794, -0.9997855329780765, 0.9997714113836412, -0.9997575245388718, 0.99974388564809, -0.9997305076630332, 0.9997174032705554, -0.9997045848806279, 0.9996920646145915, -0.9996798542937213, 0.9996679654280757, -0.999656409205673, 0.999645196481973, -0.9996343377697074, 0.9996238432290363, -0.9996137226580682, 0.9996039854837299, -0.9995946407530076, 0.999585697124568, -0.9995771628607526, 0.9995690458199766, -0.9995613534495146, 0.9995540927786966, -0.9995472704125142, 0.9995408925256386, -0.9995349648568729, 0.9995294927040126, -0.9995244809191581, 0.9995199339044453, -0.9995158556082276, 0.999512249521693, -0.9995091186759331, 0.9995064656394507, -0.9995042925161308, 0.9995026009436473, -0.9995013920923359, 

## 2.5 Computer problem 6b

```
Number of steps: 5403

Backward Error: 4.836575538957888e-7
```


In [10]:
function f(u, v, w)
    return [2u^2 − 4u + v^2 + 3w^2 + 6w + 2,  u^2 + v^2 − 2v + 2w^2 − 5, 3u^2 − 12u + v^2 + 3w^2 + 8]
end

f (generic function with 1 method)

In [11]:
function jacobian_f(u, v, w)
    return [4u - 4 2v 6w + 6; 2u 2v - 2 4w; 6u - 12 2v 6w]
end

jacobian_f (generic function with 1 method)

In [12]:
function newtons(f, jacobian, x0, linear_solver, tol=0.5 * 10^-6, linear_tol::Float64=0.5*10^-2)
    x = x0
    xLast = x0
    iterations = 0
    
    while iterations == 0 || norm(xLast - x, Inf) > tol
        x1 = copy(x)

        s = linear_solver(jacobian(x...), f(x...))

        xLast = x
        x = x1

        iterations += 1
    end

    return x, iterations
end

newtons (generic function with 3 methods)

In [13]:
gaussian(A, b) = A\b
x1, iterations1 = newtons(f, jacobian_f, ones(3), gaussian)
x2, iterations2 = newtons(f, jacobian_f, ones(3)*10, gaussian)


Markdown.parse("""
    ## Section 2.7 - Computer Problem 4
    
    ``x_1 = $(x1)``: Calculated in  $(iterations1) iterations

    ``f(x_1) =`` $(f(x1...))
    
    ``x_2 = $(x2)``: Calculated in  $(iterations2) iterations

    ``f(x_2) =`` $(f(x2...))
    

    Gaussian Elimination performs better as a linear solver than Jacobian and Gauss Sidel.
    """)

## Section 2.7 - Computer Problem 4

$x_1 = [1.0, 1.0, 1.0]$: Calculated in  1 iterations

$f(x_1) =$ [10.0, -3.0, 3.0]

$x_2 = [10.0, 10.0, 10.0]$: Calculated in  1 iterations

$f(x_2) =$ [622.0, 375.0, 588.0]

Gaussian Elimination performs better as a linear solver than Jacobian and Gauss Sidel.
